# 🚀 1-CLICK BULLETPROOF BANGLA AI OCR (COLAB GPU)

This is a **100% self-contained, single-click** training script.

### Instructions:
1. In Google Colab top menu: **Runtime ➔ Change runtime type ➔ T4 GPU ➔ Save**.
2. On the left panel (📁), upload **`dataset.zip`**.
3. Click the Play button on the single cell below! (Takes ~15-20 minutes on GPU).

In [ ]:
# =============================================================================
# 1. INSTALL STABLE PINNED DEPENDENCIES
# =============================================================================
!pip install -q --upgrade pip
!pip install -q transformers==4.44.2 accelerate==0.34.2 sentencepiece protobuf tiktoken datasets evaluate jiwer

import os, sys, zipfile, json, random, time, torch
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset
from transformers import (
    AutoImageProcessor,
    RobertaTokenizer,
    TrOCRProcessor,
    VisionEncoderDecoderModel,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    default_data_collator
)
import evaluate

print(f"⚡ PyTorch Version: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🔥 GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ WARNING: GPU is not enabled! Please go to Runtime ➔ Change runtime type ➔ T4 GPU!")

# =============================================================================
# 2. UNZIP DATASET
# =============================================================================
zip_candidates = ['/content/dataset.zip', '/content/project_data.zip', 'dataset.zip']
found_zip = None
for z in zip_candidates:
    if os.path.exists(z):
        found_zip = z
        break

if found_zip:
    print(f"📦 Unzipping {found_zip} to /content/dataset...")
    os.makedirs('/content/dataset', exist_ok=True)
    with zipfile.ZipFile(found_zip, 'r') as zip_ref:
        zip_ref.extractall('/content/dataset')
    print("✅ Unzip complete!")
else:
    print("⚠️ dataset.zip not found in root. Checking if /content/dataset already exists...")

# Locate train.jsonl
dataset_dir = Path('/content/dataset')
train_json = None
for p in dataset_dir.rglob('train.jsonl'):
    train_json = p
    break

if not train_json:
    raise FileNotFoundError("❌ ERROR: Please upload dataset.zip to the left file panel (📁) and re-run!")

DATASET_ROOT = train_json.parent
print(f"🎯 Dataset located at: {DATASET_ROOT}")

# =============================================================================
# 3. DATASET CLASS & METRICS
# =============================================================================
class BanglaOCRDataset(Dataset):
    def __init__(self, jsonl_file: Path, processor: TrOCRProcessor, max_target_length: int = 128):
        self.data_dir = jsonl_file.parent
        self.processor = processor
        self.max_target_length = max_target_length
        self.records = [json.loads(line) for line in open(jsonl_file, 'r', encoding='utf-8') if line.strip()]

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        item = self.records[idx]
        img_p = self.data_dir / item["file_name"]
        try:
            image = Image.open(img_p).convert("RGB")
        except Exception:
            image = Image.new("RGB", (384, 48), color="white")
            
        pixel_values = self.processor(image, return_tensors="pt").pixel_values.squeeze(0)
        labels = self.processor.tokenizer(item["text"], padding="max_length", max_length=self.max_target_length, truncation=True).input_ids
        labels = [l if l != self.processor.tokenizer.pad_token_id else -100 for l in labels]
        return {"pixel_values": pixel_values, "labels": torch.tensor(labels)}

cer_metric = evaluate.load("cer")
wer_metric = evaluate.load("wer")

def compute_metrics(pred, processor):
    labels_ids = pred.label_ids
    pred_ids = pred.predictions
    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    labels_ids[labels_ids == -100] = processor.tokenizer.pad_token_id
    label_str = processor.batch_decode(labels_ids, skip_special_tokens=True)
    return {"cer": cer_metric.compute(predictions=pred_str, references=label_str), "wer": wer_metric.compute(predictions=pred_str, references=label_str)}

# =============================================================================
# 4. LOAD MODEL & TOKENIZER (EXPLICIT STABLE LOAD)
# =============================================================================
print("\n🤖 Loading TrOCR Architecture...")
model_name = "microsoft/trocr-base-handwritten"
image_processor = AutoImageProcessor.from_pretrained(model_name)
tokenizer = RobertaTokenizer.from_pretrained(model_name)
processor = TrOCRProcessor(image_processor=image_processor, tokenizer=tokenizer)

model = VisionEncoderDecoderModel.from_pretrained(model_name)
model.config.decoder_start_token_id = tokenizer.cls_token_id
model.config.pad_token_id = tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size

# Datasets
train_ds = BanglaOCRDataset(DATASET_ROOT / "train.jsonl", processor)
val_json = DATASET_ROOT / "val.jsonl" if (DATASET_ROOT / "val.jsonl").exists() else DATASET_ROOT / "train.jsonl"
val_ds = BanglaOCRDataset(val_json, processor)
print(f"📊 Train samples: {len(train_ds)} | Val samples: {len(val_ds)}")

# =============================================================================
# 5. START GPU TRAINING LOOP
# =============================================================================
OUTPUT_DIR = "/content/bangla_ocr_trained_model"

training_args = Seq2SeqTrainingArguments(
    predict_with_generate=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=8 if torch.cuda.is_available() else 2,
    per_device_eval_batch_size=8 if torch.cuda.is_available() else 2,
    fp16=torch.cuda.is_available(),
    output_dir=OUTPUT_DIR,
    logging_steps=20,
    num_train_epochs=3,
    learning_rate=5e-5,
    weight_decay=0.01,
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    processing_class=processor,
    args=training_args,
    compute_metrics=lambda p: compute_metrics(p, processor),
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=default_data_collator,
)

print("\n🔥 STARTING TRAINING LOOP ON GPU...")
start_t = time.time()
trainer.train()
print(f"\n🎉 Training completed in {(time.time()-start_t)/60:.2f} minutes!")

# =============================================================================
# 6. SAVE & ZIP FINAL MODEL
# =============================================================================
final_save_dir = Path(OUTPUT_DIR) / "final_model"
final_save_dir.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(final_save_dir))
processor.save_pretrained(str(final_save_dir))

!zip -r /content/my_trained_bangla_ocr.zip /content/bangla_ocr_trained_model/final_model
print("\n🏆 DONE! Download your trained model from left panel: /content/my_trained_bangla_ocr.zip")